# 01 — Data Cleaning & Reconciliation

Source: four separate workbook exports of the Maji Ndogo water-services tracking system
(`Md_water_services_data.xlsx`, `... Full data.xlsx`, `... till 2024.xlsx`, `... part 4.xlsx`),
covering water sources, site visits, well-water quality tests, infrastructure improvement
projects, vendors, and water-source-related crime across 5 provinces and 31 towns.

**This notebook's main finding, before any business analysis starts: none of the four exports
is individually complete.** Each is missing different columns/tables that another export has.
The cleaning task here is a genuine reconciliation exercise, not a simple duplicate-drop — this
notebook documents exactly which export supplied which field, and why, so the merge is
auditable rather than an arbitrary "pick a file" decision.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 140)

# Raw exports are not committed to this repo (see README) -- these paths point
# to the original ALX coursework export location on the machine this was built on.
FULL = "C:/Users/Administrator/Downloads/ALX Coursework/Datasets/Md_water_services_data Full data.xlsx"
PLAIN = "C:/Users/Administrator/Downloads/ALX Coursework/Datasets/Md_water_services_data.xlsx"

full_sheets = pd.ExcelFile(FULL).sheet_names
plain_sheets = pd.ExcelFile(PLAIN).sheet_names
print("Sheets in 'Full data' export:", full_sheets)
print("Sheets in plain export:      ", plain_sheets)


Sheets in 'Full data' export: ['well_pollution', 'water_source', 'visits', 'queue_composition', 'project_progress', 'vendors', 'location', 'infrastructure_cost', 'water_source_related_crime']
Sheets in plain export:       ['well_pollution', 'water_source', 'visits', 'queue_composition', 'project_progress', 'location', 'infrastructure_cost', 'water_source_related_crime']


## Column-level completeness audit

Rather than assume one file is "the" source of truth, every shared table was checked column
by column for which export actually has non-null values.


In [2]:
checks = {
    "well_pollution": ["date", "description", "pollutant_ppm", "biological"],
    "location": ["address"],
    "visits": ["time_of_record", "assigned_employee_id"],
    "project_progress": ["address", "province", "comments"],
    "water_source_related_crime": ["crime_id", "time_of_incident"],
}

for table, cols in checks.items():
    full_df = pd.read_excel(FULL, sheet_name=table)
    plain_df = pd.read_excel(PLAIN, sheet_name=table)
    print(f"--- {table} ---")
    for col in cols:
        full_notna = full_df[col].notna().sum() if col in full_df.columns else "missing"
        plain_notna = plain_df[col].notna().sum() if col in plain_df.columns else "missing"
        print(f"  {col}: Full data non-null={full_notna}, plain non-null={plain_notna}")


--- well_pollution ---
  date: Full data non-null=0, plain non-null=17383
  description: Full data non-null=0, plain non-null=17383
  pollutant_ppm: Full data non-null=0, plain non-null=17383
  biological: Full data non-null=0, plain non-null=17383
--- location ---
  address: Full data non-null=0, plain non-null=39650
--- visits ---
  time_of_record: Full data non-null=0, plain non-null=60146
  assigned_employee_id: Full data non-null=0, plain non-null=60146
--- project_progress ---
  address: Full data non-null=0, plain non-null=25398
  province: Full data non-null=0, plain non-null=25398
  comments: Full data non-null=0, plain non-null=0
--- water_source_related_crime ---
  crime_id: Full data non-null=0, plain non-null=77226
  time_of_incident: Full data non-null=0, plain non-null=77226


**Result:** `well_pollution`'s detail columns, `visits.time_of_record` /
`assigned_employee_id`, and `project_progress`'s `address`/`province`/`comments` are all
entirely null in the "Full data" export but populated in the plain export. Meanwhile the plain
export is missing the `vendors` table entirely and has a less complete `project_progress`
(no vendor/cost/completion-date columns) — those only exist in "Full data". A closer look at
`project_progress` also shows *why*: the plain export's `source_status` for a sample project is
`"Backlog"` with no completion date, while the same `project_id` in "Full data" shows
`"Complete"` with a real completion date and cost. **The plain export is an earlier snapshot of
the same tracking system, not a competing version of the truth** — so static fields (address,
province) are safely backfilled from it, but dynamic fields (status, cost, dates) must come
from the newer "Full data" export.

In [3]:
project_id = 1
print("Full data snapshot:")
print(pd.read_excel(FULL, sheet_name="project_progress").query("project_id == @project_id")[["project_id","source_status","date_of_completion","cost"]])
print()
print("Plain (earlier) snapshot:")
print(pd.read_excel(PLAIN, sheet_name="project_progress").query("project_id == @project_id")[["project_id","source_status","date_of_completion"]])


Full data snapshot:
   project_id source_status date_of_completion     cost
0           1      Complete         2023-01-07  9296.52

Plain (earlier) snapshot:
   project_id source_status  date_of_completion
0           1       Backlog                 NaN


## Building the reconciled tables

In [4]:
water_source = pd.read_excel(FULL, sheet_name="water_source")

# location: town_name granularity from Full data (matches the province map's
# town boundaries, e.g. "Harare-A"); address backfilled from the plain export.
location = pd.read_excel(FULL, sheet_name="location")
location_addr = pd.read_excel(PLAIN, sheet_name="location")[["location_id", "address"]]
location = location.drop(columns=["address"]).merge(location_addr, on="location_id", how="left")

# visits: backfill the two entirely-null columns from the plain export
visits = pd.read_excel(FULL, sheet_name="visits")
visits_fill = pd.read_excel(PLAIN, sheet_name="visits")[["record_id", "time_of_record", "assigned_employee_id"]]
visits = visits.drop(columns=["time_of_record", "assigned_employee_id"]).merge(visits_fill, on="record_id", how="left")

vendors = pd.read_excel(FULL, sheet_name="vendors")

# project_progress: keep Full data's status/cost/vendor/dates (the current
# truth), backfill only the three static fields from the earlier snapshot
project_progress = pd.read_excel(FULL, sheet_name="project_progress")
pp_fill = pd.read_excel(PLAIN, sheet_name="project_progress")[["project_id", "address", "province", "comments"]]
project_progress = project_progress.drop(columns=["address", "province", "comments"]).merge(pp_fill, on="project_id", how="left")

infrastructure_cost = pd.read_excel(FULL, sheet_name="infrastructure_cost").dropna()

# well_pollution, queue_composition, water_source_related_crime: the plain
# export has real values; Full data's versions of these are empty
well_pollution = pd.read_excel(PLAIN, sheet_name="well_pollution")
queue_composition = pd.read_excel(PLAIN, sheet_name="queue_composition").drop(columns=["Unnamed: 0"], errors="ignore")
water_source_related_crime = pd.read_excel(PLAIN, sheet_name="water_source_related_crime").drop(columns=["Column1", "Unnamed: 0"], errors="ignore")

print("Reconciled table shapes:")
for name, df in [("water_source", water_source), ("location", location), ("visits", visits),
                  ("vendors", vendors), ("project_progress", project_progress),
                  ("infrastructure_cost", infrastructure_cost), ("well_pollution", well_pollution),
                  ("queue_composition", queue_composition), ("water_source_related_crime", water_source_related_crime)]:
    print(f"  {name}: {df.shape}, nulls remaining: {df.isna().sum().sum()}")


Reconciled table shapes:
  water_source: (39650, 3), nulls remaining: 0
  location: (39650, 5), nulls remaining: 0
  visits: (60146, 7), nulls remaining: 0
  vendors: (95, 4), nulls remaining: 0
  project_progress: (25398, 13), nulls remaining: 25398
  infrastructure_cost: (12, 2), nulls remaining: 0
  well_pollution: (17383, 6), nulls remaining: 0
  queue_composition: (60146, 4), nulls remaining: 0
  water_source_related_crime: (77226, 5), nulls remaining: 0


`project_progress.comments` is still 100% null after backfill — checked directly against the
raw plain export (not just the merge) and it is genuinely empty in every source file, so it is
dropped rather than kept as a column of nothing.

In [5]:
project_progress = project_progress.drop(columns=["comments"])
print("comments dropped. Final project_progress columns:", list(project_progress.columns))


comments dropped. Final project_progress columns: ['project_id', 'source_id', 'town', 'source_type', 'improvement', 'source_status', 'date_of_completion', 'assigned_vendor', 'date_started', 'cost', 'address', 'province']


## Referential integrity & duplicate checks

In [6]:
print("Duplicate source_ids:", water_source["source_id"].duplicated().sum())
print("Duplicate location_ids:", location["location_id"].duplicated().sum())

# source_id has no explicit FK column to location_id -- discovered empirically
# that source_id = location_id + a 3-character source-type suffix
test = water_source["source_id"].str[:-3]
unmatched = (~test.isin(location["location_id"])).sum()
print(f"source_id -> location_id via [:-3] suffix rule: {unmatched} unmatched out of {len(water_source)}")

visits_unmatched_loc = (~visits["location_id"].isin(location["location_id"])).sum()
visits_unmatched_src = (~visits["source_id"].isin(water_source["source_id"])).sum()
print(f"visits with unmatched location_id: {visits_unmatched_loc}")
print(f"visits with unmatched source_id: {visits_unmatched_src}")


Duplicate source_ids: 0
Duplicate location_ids: 0
source_id -> location_id via [:-3] suffix rule: 0 unmatched out of 39650
visits with unmatched location_id: 0
visits with unmatched source_id: 0


## Time-period mismatch — a limitation to flag, not paper over

`water_source_related_crime` covers **2010–2020**. `visits` (site-visit queue records) covers
**2021–2023**. These do not overlap by even a single day. Any analysis relating crime to queueing
patterns can only be done at the *location* level (which towns/provinces), never at the *time*
level — the two datasets describe different eras of the same places, not the same period.

In [7]:
print("Crime data range:", water_source_related_crime['time_of_incident'].min(), "to", water_source_related_crime['time_of_incident'].max())
print("Visits data range:  ", visits['time_of_record'].min(), "to", visits['time_of_record'].max())


Crime data range: 2010-01-01 04:00:00 to 2020-12-31 23:00:00
Visits data range:   2021-01-01 09:10:00 to 2023-07-14 13:53:00


## Export cleaned tables

In [8]:
import sqlite3, os

out_dir = "../data"
for name, df in [("water_source", water_source), ("location", location), ("visits", visits),
                  ("vendors", vendors), ("project_progress", project_progress),
                  ("infrastructure_cost", infrastructure_cost), ("well_pollution", well_pollution),
                  ("queue_composition", queue_composition), ("water_source_related_crime", water_source_related_crime)]:
    df.to_csv(f"{out_dir}/{name}.csv", index=False)

db_path = f"{out_dir}/water_services.db"
if os.path.exists(db_path):
    os.remove(db_path)
conn = sqlite3.connect(db_path)
for name, df in [("location", location), ("vendors", vendors), ("infrastructure_cost", infrastructure_cost),
                  ("water_source", water_source), ("visits", visits), ("queue_composition", queue_composition),
                  ("well_pollution", well_pollution), ("water_source_related_crime", water_source_related_crime),
                  ("project_progress", project_progress)]:
    df.to_sql(name, conn, if_exists="replace", index=False)
conn.close()

print("Exported 9 cleaned CSVs + water_services.db")
print(f"Total population represented: {water_source['number_of_people_served'].sum():,}")


Exported 9 cleaned CSVs + water_services.db
Total population represented: 27,628,140
